# 2.1.8 — Execution Architecture and Native Numerical Kernels

**Module:** 02 — NumPy: Advanced Numerical Computing and Array Programming  
**Chapter:** 2.1 — NumPy Architecture and Fundamentals  
**Level:** Expert / Research-Oriented Foundations  
**Prerequisites:** Advanced Python object semantics; basic linear algebra; numerical analysis fundamentals.

> **Purpose.** Develop a precise, testable conceptual and computational model of execution architecture and native numerical kernels. This notebook separates mathematical intent, Python-level semantics, storage consequences, and execution behavior. It is a standalone section of the Chapter 2.1 architecture sequence.

## 1. Learning Objectives

After studying this notebook, you should be able to:

1. State the central abstractions and identify the level of the system at which each guarantee applies.
2. Explain the relevant mathematical or computational invariants without relying on informal analogies.
3. Construct executable tests that distinguish correct behavior from plausible but invalid assumptions.
4. Identify boundary cases, side effects, dtype concerns, and complexity implications.
5. Translate the topic into a reproducible, documented numerical research workflow.

## 2. Reproducibility and Environment

The examples use the standard `numpy` package and Python standard library. They are intended for recent NumPy 2.x installations; most examples also work on current 1.26 releases, but NumPy 2.x promotion and interface details should be interpreted with the installed version in mind. Execute cells **top to bottom**. There are no external datasets, downloads, or network dependencies.

In [ ]:
import numpy as np
import sys

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
assert tuple(map(int, np.__version__.split('.')[:2])) >= (1, 24), 'Use a recent NumPy release.'

## 3. Conceptual Foundations and Formal Semantics

### 1. Dispatch stages

A typical numerical operation starts with Python evaluating operands and calling an array operation. NumPy checks array types and shapes, selects type-dependent loops or suitable library implementations, prepares outputs and operands, and executes the computation. The exact internal path depends on operation, dtype, layout, subclass dispatch, and build.

### 2. Ufuncs and native loops

A ufunc is an operation family capable of applying elementwise kernels to array inputs, including broadcasting, casting, and optional output placement. Native execution reduces repeated Python bytecode dispatch; it does not imply every operand uses SIMD or every operation releases the GIL.

### 3. BLAS and LAPACK boundaries

Dense matrix multiplication and decompositions can invoke optimized external numeric libraries. Their performance depends on problem size, memory layout, cache behavior, thread configuration, and installed binary. Small arrays may be dominated by dispatch overhead even when the larger operation is fast.

### 4. Data-dependent paths

Object DTypes may invoke Python-level object operations; custom array subclasses or protocols may route operations elsewhere. Noncontiguous arrays can require strided loops or temporary contiguous buffers. A correct high-level performance analysis must accommodate such dispatch differences.

### 5. Measurement discipline

Benchmarks should warm up, repeat executions, avoid measuring object construction unless intentional, and report hardware/library provenance. Microbenchmark results are not universal complexity results. Distinguish throughput from latency and compute-bound from bandwidth-bound workloads.

## 4. Architectural Model and Invariants

A useful four-layer checklist for this topic:

| Layer | Question to ask | Representative invariant |
|---|---|---|
| Logical | What operation is mathematically intended? | Axes and result shape correspond to the model |
| Representation | What dtype and storage model represent the inputs? | No unsupported casting or unexpected aliasing |
| Execution | Which category of operation processes the arrays? | Results are stable under a reference implementation |
| Validation | What independent evidence establishes correctness? | Assertions, boundary tests, and numerical tolerances |

**Important:** Shape agreement and code execution are not sufficient evidence of mathematical validity. Conversely, a mathematically valid expression can be operationally impractical or numerically unstable.

## 5. Executable Investigations

Each investigation is independent in purpose; execute in sequence so shared imports are available.

### 5.1 — Inspect ufunc metadata

**Research question.** What does this experiment establish, and which conclusion would be unjustified if we only inspected its printed values?

In [ ]:
import numpy as np
print('ufunc type:', type(np.add))
print('nin/nout:', np.add.nin, np.add.nout)
print('sample signatures:', np.add.types[:8])
assert np.add.nin == 2 and np.add.nout == 1

### 5.2 — Separating allocation from work

**Research question.** What does this experiment establish, and which conclusion would be unjustified if we only inspected its printed values?

In [ ]:
n = 100_000
x = np.arange(n, dtype=np.float64)
y = np.ones_like(x)
out = np.empty_like(x)
np.add(x, y, out=out)
np.testing.assert_allclose(out, x + 1)
print('array bytes:', x.nbytes, 'output reused:', out.shape)

### 5.3 — Repeatable local timing, not a universal claim

**Research question.** What does this experiment establish, and which conclusion would be unjustified if we only inspected its printed values?

In [ ]:
from time import perf_counter
x = np.arange(200_000, dtype=np.float64)
out = np.empty_like(x)
for _ in range(3): np.sqrt(x, out=out)
measurements = []
for _ in range(7):
    start = perf_counter()
    np.sqrt(x, out=out)
    measurements.append(perf_counter() - start)
print('median seconds:', float(np.median(measurements)))

## 6. Edge Cases, Failure Modes, and Misconceptions

- **Logical validity is not physical contiguity.** Shape alone cannot determine whether an operation needs a copy.
- **Value equality is not identity.** Two arrays may compare equal while having independent storage, or share storage while being distinct Python objects.
- **A vectorized expression does not certify correctness.** Broadcasting and axis conventions can silently encode unintended mathematics.
- **A dtype is not a real-number model.** Overflow, finite precision, and representation-dependent transformations require explicit attention.
- **A benchmark does not imply a theorem.** Hardware, build configuration, allocation strategy, and input scale can affect conclusions.
- **Array semantics do not encode domain assumptions.** For finance or statistics, look-ahead restrictions, missing-data rules, weights, and alignment remain external responsibilities.

When evaluating an unexpected result, first inspect `shape`, `dtype`, `strides`, and storage overlap, then isolate the mathematical operation on a hand-checkable example.

## 7. Performance and Complexity Considerations

The performance question should be asked in three parts: **work complexity**, **data movement**, and **peak live memory**. A contiguous numeric array can support efficient native processing, but a particular operation may be limited by memory bandwidth, temporary allocation, scalar dispatch, or native-library parallelism. Use measurements only after verifying correctness, and never assume that a microbenchmark generalizes to different dtypes or problem sizes.

A practical benchmark should (i) isolate the operation, (ii) distinguish input setup from computation, (iii) warm up the relevant execution path, (iv) collect repeated measurements, and (v) record platform and NumPy provenance.

## 8. Analytical Interpretation

The high-level expression is a *request* to an execution machinery, not a guarantee about the specific kernel. Performance claims must be scoped to a dtype, shape, allocation policy, layout, hardware, and numerical library.

## 9. Advanced Exercises

These are open-ended expert exercises. Provide an analytical justification, a minimal counterexample where useful, and an executable validation test.

**Exercise 1.** Describe the full dispatch path of a broadcasted float64 `np.add`, including possible allocations.

**Exercise 2.** Why is a `dtype=object` benchmark not representative of numeric SIMD performance?

**Exercise 3.** Devise an experiment to detect thread oversubscription in matrix multiplication without assuming a specific BLAS vendor.

## 10. Expected Results and Validation Strategy

1. Execute every example in order; all assertions should pass on an appropriately configured NumPy installation.
2. Compare printed shapes, types, and DTypes with the surrounding mathematical and architectural claims.
3. Where a local benchmark is included, record measured values but **do not** expect a universal numeric timing.
4. Modify input shapes, dtypes, and views to construct at least one failing case; document why the original contract no longer holds.
5. Implement an independent reference for one operation and compare using an error metric appropriate to its numerical domain.

**Acceptance criteria:** the student can explain the behavior without relying on a copied output, design a counterexample to an invalid inference, and describe relevant representation and performance limitations.

## Further Reading and Verification Sources

The following are authoritative public references. API details may vary across NumPy versions; verify against the installed version when reproducing an experiment.

- [NumPy: The N-dimensional array](https://numpy.org/doc/stable/reference/arrays.ndarray.html)
- [NumPy: ndarray reference](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.html)
- [NumPy: Data types](https://numpy.org/doc/stable/reference/arrays.dtypes.html)
- [NumPy: Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html)
- [NumPy: Copies and views](https://numpy.org/doc/stable/user/basics.copies.html)
- [NumPy: Universal functions](https://numpy.org/doc/stable/reference/ufuncs.html)
- [NumPy: Thread safety](https://numpy.org/doc/stable/reference/thread_safety.html)
- [NumPy: C API](https://numpy.org/doc/stable/reference/c-api/index.html)

**Scope note:** This is Chapter 2.1 (architectural foundations). Detailed DType implementation, advanced indexing, stride tricks, microarchitecture, and native C extension development are treated in Chapters 2.3, 2.5, 2.7, 2.21 and 2.23 respectively.